# Phase 3b - Optimizer ablation (Colab T4)

Compares **SGD / Adam / AdamW** at `imgsz=640` (the Phase 3a winner), loss
weights held fixed.

## Each optimizer gets its own learning rate - on purpose

Holding one `lr0` across optimizers looks fair but isn't: it just measures
which optimizer suits that particular rate. Our Phase 2 sweep shows how big
the effect is - **AdamW at lr0=0.01 reaches only ~0.24 mAP50, the same
optimizer at 8.8e-4 reaches 0.50**. Ultralytics agrees: its `auto` mode
picks (SGD, 0.01) or (AdamW, 0.002*5/(4+nc)).

So the comparison unit is **optimizer + the rate appropriate to it**:

| | SGD | Adam / AdamW |
|---|---|---|
| hazard | 0.01 | 8.8e-4 (Phase 2 tuned) |
| child | 0.01 | 0.002 (ultralytics auto, nc=1) |

State this in the write-up rather than claiming a single-variable change.

**Budget:** ~2.3 h hazard + ~2.0 h child at 30 epochs. Run one model per
session. Resumable - re-run the same cell to continue after a disconnect.

In [ ]:
!nvidia-smi

In [ ]:
!pip install -q ultralytics==8.4.106 roboflow

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
ABL_PROJECT = "/content/drive/MyDrive/deeplrn_group2/runs"
import os; os.makedirs(ABL_PROJECT, exist_ok=True)
print("runs ->", ABL_PROJECT)

In [ ]:
import os
REPO_URL = "https://github.com/FooJames/DEEPLRN_Group2.git"
if not os.path.isdir("DEEPLRN_Group2"):
    !git clone $REPO_URL
else:
    !cd DEEPLRN_Group2 && git pull
%cd DEEPLRN_Group2

In [ ]:
from google.colab import userdata
import os
os.environ["ROBOFLOW_API_KEY"] = userdata.get("ROBOFLOW_API_KEY")
print("key loaded:", bool(os.environ.get("ROBOFLOW_API_KEY")))

In [ ]:
!python scripts/download_data.py --child-version 3 --hazard-version 1
!python scripts/fix_data_yaml.py data/child/data.yaml data/hazard/data.yaml
!python scripts/normalize_child_labels.py data/child

## Session A - hazard (~2.3 h)

Uses the Phase 2 tuned loss weights (box/cls/dfl) fixed across all three
optimizers. Re-run to resume.

In [ ]:
!python scripts/ablation_optimizer.py --model hazard --data data/hazard/data.yaml     --imgsz 640 --epochs 30 --project "$ABL_PROJECT"

In [ ]:
!zip -r ablation_opt_hazard.zip results/metrics/ablation_optimizer_hazard.csv "$ABL_PROJECT"/ablation_opt_hazard_*
!unzip -l ablation_opt_hazard.zip | tail -5
from google.colab import files
files.download("ablation_opt_hazard.zip")

## Session B - child (~2.0 h)

Child was never tuned, so loss weights stay at defaults and the adaptive
rate comes from ultralytics' auto formula (0.002 for nc=1).

In [ ]:
!python scripts/ablation_optimizer.py --model child --data data/child/data.yaml     --imgsz 640 --epochs 30 --project "$ABL_PROJECT"

In [ ]:
!zip -r ablation_opt_child.zip results/metrics/ablation_optimizer_child.csv "$ABL_PROJECT"/ablation_opt_child_*
!unzip -l ablation_opt_child.zip | tail -5
from google.colab import files
files.download("ablation_opt_child.zip")

## Results

In [ ]:
import pandas as pd, os
for m in ("hazard", "child"):
    p = f"results/metrics/ablation_optimizer_{m}.csv"
    if os.path.isfile(p):
        print(f"--- {m} ---")
        print(pd.read_csv(p)[["optimizer","lr0","mAP50","mAP50_95",
                              "train_min","infer_ms"]].to_string(index=False), "
")
print("Phase 3a @640/30ep for reference: hazard 0.5259 | child 0.9338 (both AdamW)")

### Notes
- AdamW here should roughly reproduce the Phase 3a @640 number (same
  optimizer, same rate, same budget) - a useful consistency check. Small
  differences are run-to-run variance.
- `auto` is deliberately rejected by the script: it overrides `lr0` and
  chooses the optimizer itself, so it is not a comparable condition.
- Val split only; the test split stays untouched until the end.